In [7]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

# ---------------------- STEP 1: Load Training Data ---------------------- #
train_df = pd.read_csv("maintaince_record table.csv")
train_df["maintenance_date"] = pd.to_datetime(train_df["maintenance_date"])
train_df = train_df.sort_values(by=["printer_id", "maintenance_date"])

# ---------------------- STEP 2: Feature Engineering ---------------------- #
train_df["days_since_last"] = train_df.groupby("printer_id")["maintenance_date"].diff().dt.days
train_df["maintenance_count"] = train_df.groupby("printer_id").cumcount() + 1
train_df["avg_cost"] = train_df.groupby("printer_id")["cost"].transform("mean")
train_df["days_since_last"] = train_df["days_since_last"].fillna(train_df["days_since_last"].median())
train_df["maintenance_needed_soon"] = (train_df["days_since_last"] < 30).astype(int)

# ---------------------- STEP 3: Encode Categorical ---------------------- #
train_df = pd.get_dummies(train_df, columns=["description"], prefix="issue")
train_df = train_df.drop(columns=["record_id", "maintenance_date", "technician"], errors="ignore")

# ---------------------- STEP 4: Train Model ---------------------- #
latest_entries = train_df.groupby("printer_id").tail(1)
historical_entries = train_df.drop(latest_entries.index)

X_train = historical_entries.drop(columns=["maintenance_needed_soon", "printer_id"])
y_train = historical_entries["maintenance_needed_soon"]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Save model
joblib.dump(model, "maintenance_predictor_model.pkl")
print("✅ Model saved as 'maintenance_predictor_model.pkl'")

# ---------------------- STEP 5: Load Test Data ---------------------- #
test_df = pd.read_csv(r"c:\Users\Hp admin\Downloads\testdata_maintainance.csv")
test_df["maintenance_date"] = pd.to_datetime(test_df["maintenance_date"])
test_df = test_df.sort_values(by=["printer_id", "maintenance_date"])

test_df["days_since_last"] = test_df.groupby("printer_id")["maintenance_date"].diff().dt.days
test_df["maintenance_count"] = test_df.groupby("printer_id").cumcount() + 1
test_df["avg_cost"] = test_df.groupby("printer_id")["cost"].transform("mean")
test_df["days_since_last"] = test_df["days_since_last"].fillna(test_df["days_since_last"].median())

test_df = pd.get_dummies(test_df, columns=["description"], prefix="issue")
test_df_clean = test_df.drop(columns=["record_id", "maintenance_date", "technician"], errors="ignore")

# ---------------------- STEP 6: Align Features ---------------------- #
expected_features = model.feature_names_in_
for col in expected_features:
    if col not in test_df_clean.columns:
        test_df_clean[col] = 0
test_df_clean = test_df_clean[expected_features]

# ---------------------- STEP 7: Predict ---------------------- #
test_df["maintenance_predicted"] = model.predict(test_df_clean)

# Add classification logic for printer condition
def classify_printer(gap):
    if gap < 30:
        return "Bad"
    elif gap < 70:
        return "Average"
    else:
        return "Good"

# Apply the classify_printer function based on the predicted gap
test_df["maintenance_predicted_label"] = test_df["days_since_last"].apply(classify_printer)

# Save predictions
test_df.to_csv("maintenance_predictions_output.csv", index=False)
print("✅ Predictions saved to 'maintenance_predictions_output.csv'")

# Print only unique printer IDs predicted to need maintenance
at_risk_printers = test_df[test_df["maintenance_predicted"] == 1]["printer_id"].unique()
print("\nPrinters predicted by Random Forest to need maintenance")
print(list(at_risk_printers))


✅ Model saved as 'maintenance_predictor_model.pkl'
✅ Predictions saved to 'maintenance_predictions_output.csv'

Printers predicted by Random Forest to need maintenance
['PRN026', 'PRN027', 'PRN030', 'PRN031', 'PRN033', 'PRN034', 'PRN035', 'PRN036', 'PRN038', 'PRN039', 'PRN041', 'PRN042', 'PRN043', 'PRN044', 'PRN048']
